In [1]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import cmocean.cm as cmo

In [2]:
def create_long_lat(dx, nx, ny, beta_rot, grid="T"):
    r_earth = 6370.0 # radius of the Earth
    beta    = beta_rot   # to rotate the mask with respect to Greenwich
                     # when beta = 0, x and x1 axis are aligned with Greenwich
    if grid=="T":
        if int(80 % dx)==0:
            dx_pole = 2480 + (80 - (dx / 2))  # distance of pole from tracer point of cell 0,0
            dy_pole = 2240 + (80 - (dx / 2))
        elif int(32 % dx)==0:
            dx_pole = 2464 + (32 - (dx / 2))  # distance of pole from tracer point of cell 0,0
            dy_pole = 2208 + (32 - (dx / 2))
    elif grid=="U":
        if int(80 % dx)==0:
            dx_pole = 2480 + (80 - dx)  # distance of pole from U point of cell 0,0
            dy_pole = 2240 + (80 - (dx / 2))
        elif int(32 % dx)==0:
            dx_pole = 2464 + (32 - dx)  # distance of pole from U point of cell 0,0
            dy_pole = 2208 + (32 - (dx / 2))
    elif grid=="V":
        if int(80 % dx)==0:
            dx_pole = 2480 + (80 - (dx / 2))  # distance of pole from V point of cell 0,0
            dy_pole = 2240 + (80 - dx)
        elif int(32 % dx)==0:
            dx_pole = 2464 + (32 - (dx / 2))  # distance of pole from V point of cell 0,0
            dy_pole = 2208 + (32 - dx)
    deg2rad = np.pi / 180
    rad2deg = 180 / np.pi
          
    latmin  = 90.0
    ilatmin = 0
    jlatmin = 0
    r1max   = 0
    lat = np.zeros((nx, ny))
    long = np.zeros((nx, ny))

    ii = np.arange(0, nx)
    jj = np.arange(0, ny)
    j, i = np.meshgrid(jj, ii)
    #---------- position on the stereographic plane -------------------------------
    x1 = i * dx - dx_pole
    y1 = j * dx - dy_pole
    r1 = np.sqrt(x1**2 + y1**2)
    if np.max(r1) > r1max:
        r1max = np.max(r1)
    #---------- angle of the cone -------------------------------------------------
    tanteta  = r1 / (2 * r_earth)
    #----------- short radius on the sphere ---------------------------------------
    r = 2 * r_earth * tanteta / (1 + tanteta**2)
    #----------- get the latitude and longitude -----------------------------------
    lat = np.arccos(r / r_earth) * rad2deg
    if np.min(lat < latmin):
        latmin = np.min(lat)
        ilatmin = i[np.unravel_index(lat.argmin(), lat.shape)]
        jlatmin = j[np.unravel_index(lat.argmin(), lat.shape)]
    x1 = np.where(((x1 >= 0.0) & (x1 < 1e-8)), 1e-8, x1)
    long = np.where(((x1 >= 0.0) & (y1 >= 0)), np.arctan(y1 / x1) * rad2deg + beta, long)
    long = np.where(((x1 >= 0.0) & (y1 < 0)), np.arctan(y1 / x1) * rad2deg + beta + 360.0, long)
    long = np.where((x1 < 0.0), np.arctan(y1 / x1) * rad2deg + beta + 180.0, long)
    long = np.where((long >= 360.0), long - 360.0, long)
    return long, lat

In [3]:
forcing_dir = "/storage/jrieck/SIM_forcing/"

Create old 20 km-grid and new 32 km-grid

In [4]:
dx_old = 20
nx_old = 258
ny_old = 218
beta_rot = 32.0

In [5]:
lonT_old, latT_old = create_long_lat(dx_old, nx_old+2, ny_old+2, 32.0, grid="T")

In [6]:
dx_new = 32.0
nx_new = 160
ny_new = 135
beta_rot = 32.0

In [7]:
lonT_new, latT_new = create_long_lat(dx_new, nx_new+2, ny_new+2, 32.0, grid="T")

Load 32 km mask

In [8]:
mask_tmp = np.loadtxt("/storage/jrieck/SIM_mask_bathy/mask_dx32.00_nx160_ny135.dat", dtype=str)
mask32 = np.zeros((ny_new+2, nx_new+2))
for j in range(0, ny_new+2):
    for i in range(0, nx_new+2):
        mask32[j, i] = int(mask_tmp[j][i])

Load atmospheric and oceanic temperature forcing data

In [9]:
airT_era = xr.open_dataset("/storage/jrieck/ERA5/northOf55/monthly/ERA5_2000-2025_2m_temperature.grib").roll(longitude=720, roll_coords=True)
ocnT_era = xr.open_dataset("/storage/jrieck/ERA5/northOf55/monthly/ERA5_2000-2025_sea_surface_temperature.grib").roll(longitude=720, roll_coords=True)
ocnT_era["sst"] = ocnT_era.sst.fillna(-1e12) - 273.15
periodic = xr.Dataset()
periodic["time"] = (("time"), xr.date_range(start=str(airT_era.time[0].values), end=str(airT_era.time[-1].values), freq="1MS") + pd.Timedelta(days=14))
periodic["longitude"] = (("longitude"), np.hstack([-0.5, -0.25, airT_era.longitude.where(airT_era.longitude >= 0, airT_era.longitude + 360), 360., 360.25]))
periodic["latitude"] = (("latitude"), airT_era.latitude.data)
periodic["t2m"] = (("time", "latitude", "longitude"), np.concatenate([airT_era.t2m.isel(longitude=slice(-2, -1)).values,
                                                                      airT_era.t2m.isel(longitude=slice(-1, None)).values,
                                                                      airT_era.t2m.values, 
                                                                      airT_era.t2m.isel(longitude=slice(0, 1)).values,
                                                                      airT_era.t2m.isel(longitude=slice(1, 2)).values], axis=2))
periodic["sst"] = (("time", "latitude", "longitude"), np.concatenate([ocnT_era.sst.isel(longitude=slice(-2, -1)).values,
                                                                      ocnT_era.sst.isel(longitude=slice(-1, None)).values,
                                                                      ocnT_era.sst.values, 
                                                                      ocnT_era.sst.isel(longitude=slice(0, 1)).values,
                                                                      ocnT_era.sst.isel(longitude=slice(1, 2)).values], axis=2))

interpolate to 32 km

In [10]:
gridT_new = xr.Dataset()
gridT_new["lon"] = (("y", "x"), lonT_new.T)
gridT_new["lat"] = (("y", "x"), latT_new.T)
gridT_new["mask"] = (("y", "x"), mask32)

use 2Dinterpolation from xarray (numpy/scipy)

In [388]:
airT_out = periodic.t2m.interp(longitude=gridT_new.lon, latitude=gridT_new.lat, method="pchip")

In [390]:
airT_out.to_netcdf(forcing_dir + "airT/air.mon.mean_dx" + str(dx_new) + "_nx" + str(nx_new) + "_ny" + str(ny_new) + ".nc")

In [11]:
ocnT_clim = periodic.sst.groupby("time.month").mean()
ocnT_tmp = ocnT_clim.interp(longitude=gridT_new.lon, latitude=gridT_new.lat, method="pchip")
ocnT_tmp = ocnT_tmp.where(((gridT_new.mask == 1) & (ocnT_tmp>-100)), other=np.nan)
ocnT_tmp = ocnT_tmp.interpolate_na("y", method="nearest", fill_value="extrapolate")
ocnT_out = ocnT_tmp.where(gridT_new.mask==1, other=-4)

In [395]:
for m in range(0, 12):
    l = " " + "\n ".join(["\t".join(row.astype(str)) for row in ocnT_out.isel(month=m).values])
    with open(forcing_dir + "ocnT/Tocn" + f"{m+1:02d}" + "_dx" + str(dx_new) + "_nx" + str(nx_new) + "_ny" + str(ny_new), "w") as n:
        n.write(l)

Climatological ocean currents interpolated from old forcing

In [12]:
uwater_tmp = np.loadtxt("/storage/jrieck/SIM_forcing/ocncurrent/20/uwater20.clim", dtype=None)
vwater_tmp = np.loadtxt("/storage/jrieck/SIM_forcing/ocncurrent/20/vwater20.clim", dtype=None)

In [13]:
uwater = xr.Dataset()
uwater["x"] = (("x"), -(2480 + (80 - dx_old)) + np.arange(nx_old+1) * dx_old)
uwater["y"] = (("y"), -(2240 + (80 - (dx_old / 2))) + np.arange(ny_old) * dx_old)
uwater["uwater"] = (("y", "x"), uwater_tmp)

vwater = xr.Dataset()
vwater["x"] = (("x"), -(2480 + (80 - (dx_old / 2))) + np.arange(nx_old) * dx_old)
vwater["y"] = (("y"), -(2240 + (80 - dx_old)) + np.arange(ny_old+1) * dx_old)
vwater["vwater"] = (("y", "x"), vwater_tmp)

In [14]:
uwater_out = uwater.uwater.interp(x=-(2464 + (32 - dx_new)) + np.arange(nx_new+1) * dx_new, 
                                  y=-(2208 + (32 - (dx_new / 2))) + np.arange(ny_new) * dx_new ,
                                  method="pchip")
vwater_out = vwater.vwater.interp(x=-(2464 + (32 - (dx_new / 2))) + np.arange(nx_new) * dx_new, 
                                  y=-(2208 + (32 - dx_new)) + np.arange(ny_new+1) * dx_new ,
                                  method="pchip")

In [15]:
uwater_out = uwater_out.interpolate_na("y", method="nearest", fill_value="extrapolate").interpolate_na("x", method="nearest", fill_value="extrapolate")
vwater_out = vwater_out.interpolate_na("y", method="nearest", fill_value="extrapolate").interpolate_na("x", method="nearest", fill_value="extrapolate")

In [404]:
l = " " + "\n ".join(["\t".join(row.astype(str)) for row in uwater_out.values])
with open(forcing_dir + "ocncurrent/uwater" + "_dx" + str(dx_new) + "_nx" + str(nx_new) + "_ny" + str(ny_new), "w") as n:
    n.write(l)

l = " " + "\n ".join(["\t".join(row.astype(str)) for row in vwater_out.values])
with open(forcing_dir + "ocncurrent/vwater" + "_dx" + str(dx_new) + "_nx" + str(nx_new) + "_ny" + str(ny_new), "w") as n:
    n.write(l)

Simple wind forcing (includes no rotation onto the grid, do not use for anything meaningful)

In [37]:
udata = xr.open_mfdataset("/storage/jrieck/ERA5/northOf55/ERA5_200701_10m_u_component_of_wind.grib").isel(time=slice(0, 720, 6))
vdata = xr.open_mfdataset("/storage/jrieck/ERA5/northOf55/ERA5_200701_10m_v_component_of_wind.grib").isel(time=slice(0, 720, 6))
udata["longitude"] = udata.longitude.where(udata.longitude >= 0, udata.longitude + 360)
udata = udata.sortby("longitude")
vdata["longitude"] = vdata.longitude.where(vdata.longitude >= 0, vdata.longitude + 360)
vdata = vdata.sortby("longitude")

# extend data by 1 in x direction for smooth interpolation across -180/180 boundary
# u and v from ERA5 are on the same longitude and latitude, se we use only the ones from udata
periodic = xr.Dataset()
periodic["time"] = (("time"), udata.valid_time.data)
periodic["longitude"] = (("longitude"), np.hstack([-0.5, -0.25, udata.longitude.where(udata.longitude >= 0, udata.longitude + 360), 360., 360.25]))
periodic["latitude"] = (("latitude"), udata.latitude.data)
periodic["u10"] = (("time", "latitude", "longitude"), np.concatenate([udata.u10.isel(longitude=slice(-2, -1)).values,
                                                                      udata.u10.isel(longitude=slice(-1, None)).values,
                                                                      udata.u10.values, 
                                                                      udata.u10.isel(longitude=slice(0, 1)).values,
                                                                      udata.u10.isel(longitude=slice(1, 2)).values], axis=2))
periodic["v10"] = (("time", "latitude", "longitude"), np.concatenate([vdata.v10.isel(longitude=slice(-2, -1)).values,
                                                                      vdata.v10.isel(longitude=slice(-1, None)).values,
                                                                      vdata.v10.values, 
                                                                      vdata.v10.isel(longitude=slice(0, 1)).values,
                                                                      vdata.v10.isel(longitude=slice(1, 2)).values], axis=2))

In [46]:
u_out = periodic.u10.interp(longitude=gridT_new.lon, latitude=gridT_new.lat, method="linear")
v_out = periodic.v10.interp(longitude=gridT_new.lon, latitude=gridT_new.lat, method="linear")

In [59]:
out = xr.Dataset()
out["time"] = u_out.time
out["lon"] = (("y", "x"), u_out.longitude.data)
out["lat"] = (("y", "x"), u_out.latitude.data)
out["uwnd"] = (("time", "y", "x"), u_out.data)
out["vwnd"] = (("time", "y", "x"), v_out.data)

In [63]:
out.to_netcdf(forcing_dir + "wind/DO_NOT_USE_wind_dx" + str(dx_new) + "_nx" + str(nx_new) + "_ny" + str(ny_new) + ".nc")